# IAM Role Auto-Creation via Real SDK Interfaces (No Mocks)

Calls the **actual user-facing classes** with **no role provided** and confirms the correct least-privilege IAM role is auto-created at the real entry point — not by calling the resolver util directly.

| Interface | No-role entry point | Role created | Type |
|---|---|---|---|
| `ModelTrainer` | construction (`model_post_init`) | `SageMaker-AutoRole-Training` | training |
| `SFTTrainer` | `.train()` | `SageMaker-AutoRole-Training` | training |
| `BenchMarkEvaluator` | `.evaluate()` | `SageMaker-AutoRole-Training` | training |
| `ModelBuilder` | construction (`_initialize_defaults`) | `SageMaker-AutoRole-Serving` | serving |
| `Pipeline` | `.create()` | `SageMaker-AutoRole-Pipeline` | pipeline |

> ⚠️ **Real AWS.** This creates and deletes IAM roles + policies in the active account. After each role is auto-created, the underlying operation (submit job / start eval / create pipeline) is expected to fail downstream for unrelated reasons (no real datasets/model packages) — that is fine; **the test is whether the role got created.** All roles are removed in the final cleanup cell.

> 🐍 **Kernel:** this SDK uses PEP 604 (`X | None`) typing, so it requires **Python 3.10+**. Run this notebook with the **`Python (sagemaker-recipes)`** kernel (3.10), not the default `python3` (3.9).

## Setup

In [1]:
import sys

assert sys.version_info >= (3, 10), (
    f"This notebook needs Python 3.10+ (got {sys.version.split()[0]}). "
    "Switch to the 'Python (sagemaker-recipes)' kernel."
)

for p in (
    "../sagemaker-core/src",
    "../sagemaker-train/src",
    "../sagemaker-serve/src",
    "../sagemaker-mlops/src",
):
    if p not in sys.path:
        sys.path.insert(0, p)

import logging
import boto3
from botocore.exceptions import ClientError

logging.getLogger().setLevel(logging.WARNING)  # quiet the SDK info logs

iam = boto3.client("iam")
sts = boto3.client("sts")

REGION = boto3.Session().region_name or "us-west-2"
ACCOUNT = sts.get_caller_identity()["Account"]
S3_OUTPUT = f"s3://sagemaker-{REGION}-{ACCOUNT}/iam-autocreate-test/"
MODEL_ID = "meta-textgeneration-llama-3-2-1b-instruct"  # public JumpStart base model

ALL_ROLES = [
    "SageMaker-AutoRole-Training",
    "SageMaker-AutoRole-Serving",
    "SageMaker-AutoRole-Pipeline",
    "SageMaker-AutoRole-FeatureStore",
]

print(f"Account: {ACCOUNT}")
print(f"Identity: {sts.get_caller_identity()['Arn']}")
print(f"Region:  {REGION}")
print("\n→ Confirm this is a NON-PRODUCTION account before continuing.")

Account: 211125564141
Identity: arn:aws:sts::211125564141:assumed-role/Admin/nargokul-Isengard
Region:  us-west-2

→ Confirm this is a NON-PRODUCTION account before continuing.


In [2]:
def role_exists(role_name):
    try:
        iam.get_role(RoleName=role_name)
        return True
    except ClientError:
        return False


def show_role(role_name):
    """Assert the role exists and print its attached policies."""
    assert role_exists(role_name), f"❌ Role {role_name} was NOT created!"
    attached = iam.list_attached_role_policies(RoleName=role_name)
    policies = [p["PolicyName"] for p in attached["AttachedPolicies"]]
    print(f"✓ Role auto-created: {role_name}")
    print(f"  Policies ({len(policies)}):")
    for pn in policies:
        print(f"    - {pn}")


def cleanup_role(role_name):
    try:
        for p in iam.list_attached_role_policies(RoleName=role_name).get("AttachedPolicies", []):
            iam.detach_role_policy(RoleName=role_name, PolicyArn=p["PolicyArn"])
            if ":aws:policy/" not in p["PolicyArn"]:
                try:
                    iam.delete_policy(PolicyArn=p["PolicyArn"])
                except ClientError:
                    pass
        for pn in iam.list_role_policies(RoleName=role_name).get("PolicyNames", []):
            iam.delete_role_policy(RoleName=role_name, PolicyName=pn)
        iam.delete_role(RoleName=role_name)
        print(f"  cleaned: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchEntity":
            pass
        else:
            print(f"  warn {role_name}: {e}")

## Pre-cleanup
Start fresh so every role below is genuinely created by the SDK call, not left over.

In [3]:
print("Removing any leftover auto-roles...")
for r in ALL_ROLES:
    cleanup_role(r)
print("Done.")

Removing any leftover auto-roles...
Done.


## 1. `ModelTrainer` — no role → auto-create training role at construction
`ModelTrainer(...)` with no `role` resolves the role in `model_post_init`, so just constructing it (in SageMaker training-job mode) triggers creation.

In [4]:
assert not role_exists("SageMaker-AutoRole-Training")
from sagemaker.train.model_trainer import ModelTrainer

trainer = ModelTrainer(
    training_image=f"763104351884.dkr.ecr.{REGION}.amazonaws.com/pytorch-training:2.0-cpu-py310",
    # NOTE: no role= provided
)
print(f"trainer.role = {trainer.role}\n")
show_role("SageMaker-AutoRole-Training")

/Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.1)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


[06/10/26 12:21:10] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=471091;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=742846;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /Users/nargokul/Library/Application Support/sagemaker/config.yaml


[06/10/26 12:21:11] INFO     SageMaker session not provided. Using default Session.                  ]8;id=873347;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=55843;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Creating role 'SageMaker-AutoRole-Training'...                ]8;id=154820;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=158733;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#303\303]8;;\

...

                    INFO     Compute not provided. Using default:                                   ]8;id=70661;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=535505;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py#111\111]8;;\
                             instance_type='ml.m5.xlarge' instance_count=1 volume_size_in_gb=30                    
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             instance_groups=None training_plan_arn=None                                           
                             instance_placement_config=None enable_managed_spot_training=None                      

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=747242;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=389563;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py#137\137]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

[06/10/26 12:21:24] INFO     OutputDataConfig not provided. Using default:                          ]8;id=132672;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=839513;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py#162\162]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-211125564141/pytorch-training                
                             -job' kms_key_id=None compression_type='GZIP'                                         

                    INFO     Training image URI:                                               ]8;id=249936;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=539316;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.0                     
                             -cpu-py310                                                                            

trainer.role = arn:aws:iam::211125564141:role/SageMaker-AutoRole-Training

✓ Role auto-created: SageMaker-AutoRole-Training
  Policies (6):
    - SageMaker-AutoRole-Training-cloudwatch_metric_policy
    - SageMaker-AutoRole-Training-s3_policy
    - SageMaker-AutoRole-Training-ecr_policy
    - SageMaker-AutoRole-Training-kms_policy
    - SageMaker-AutoRole-Training-cloudwatch_logs_policy
    - SageMaker-AutoRole-Training-ec2_policy


## 2. `SFTTrainer` — no role → auto-create training role on `.train()`
The role is resolved inside `.train()`. The job submission then fails downstream (placeholder dataset / model-package group) **after** the role is created — expected.

In [5]:
cleanup_role("SageMaker-AutoRole-Training")  # remove the one from step 1 to prove .train() creates it
assert not role_exists("SageMaker-AutoRole-Training")

from sagemaker.train.sft_trainer import SFTTrainer

sft = SFTTrainer(
    model=MODEL_ID,
    accept_eula=True,
    model_package_group="iam-autocreate-test-group",
    s3_output_path=S3_OUTPUT,
    # NOTE: no role= provided
)
try:
    sft.train(training_dataset=S3_OUTPUT + "train.jsonl", wait=False)
    print("train() submitted")
except Exception as e:
    print(f"(expected) train() failed downstream after role creation: {type(e).__name__}")

print()
show_role("SageMaker-AutoRole-Training")

  cleaned: SageMaker-AutoRole-Training


[06/10/26 12:21:26] INFO     SageMaker session not provided. Using default Session.                  ]8;id=693231;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=925700;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=652354;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=356351;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=24874;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=274906;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/10/26 12:21:28] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=383574;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=509139;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

...

[06/10/26 12:21:41] INFO     Resolved MLflow app:                                             ]8;id=682853;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=457137;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py#157\157]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:mlflow-app/app-X4PV75IB                      
                             XLBR (status: Created, version: 3.10.1)                                               

                    INFO     MLflow resource ARN:                                             ]8;id=122718;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=827550;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py#794\794]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:mlflow-app/app-X4PV75IB                      
                             XLBR                                                                                  

                    INFO     Auto-detecting whether dataset is multimodal:                        ]8;id=231599;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/data_utils.py\data_utils.py]8;;\:]8;id=310162;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/data_utils.py#140\140]8;;\
                             s3://sagemaker-us-west-2-211125564141/iam-autocreate-test/train.json                  
                             l                                                                                     

                    WARNING  Failed to check multimodal data from                                 ]8;id=317030;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/data_utils.py\data_utils.py]8;;\:]8;id=810335;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/data_utils.py#170\170]8;;\
                             s3://sagemaker-us-west-2-211125564141/iam-autocreate-test/train.json                  
                             l: Failed to load S3 file                                                             
                             s3://sagemaker-us-west-2-211125564141/iam-autocreate-test/train.json                  
                             l: An error occurred (NoSuchKey) when calling the GetObject                           
                             operation: The specified key does not exist.. Defaulting to                           
                             text-only.                                                                            

(expected) train() failed downstream after role creation: ClientError

✓ Role auto-created: SageMaker-AutoRole-Training
  Policies (6):
    - SageMaker-AutoRole-Training-s3_policy
    - SageMaker-AutoRole-Training-ecr_policy
    - SageMaker-AutoRole-Training-kms_policy
    - SageMaker-AutoRole-Training-cloudwatch_logs_policy
    - SageMaker-AutoRole-Training-ec2_policy
    - SageMaker-AutoRole-Training-cloudwatch_metric_policy


## 3. `BenchMarkEvaluator` — no role → auto-create training role on `.evaluate()`
The role is resolved in `_get_aws_execution_context()`, called at the start of `.evaluate()`.

In [6]:
cleanup_role("SageMaker-AutoRole-Training")
assert not role_exists("SageMaker-AutoRole-Training")

from sagemaker.train.evaluate.benchmark_evaluator import BenchMarkEvaluator, get_benchmarks

Benchmark = get_benchmarks()
evaluator = BenchMarkEvaluator(
    benchmark=Benchmark.MMLU,
    model=MODEL_ID,
    s3_output_path=S3_OUTPUT,
    evaluate_base_model=True,
    # NOTE: no role= provided
)
try:
    evaluator.evaluate()
    print("evaluate() started")
except Exception as e:
    print(f"(expected) evaluate() failed downstream after role creation: {type(e).__name__}")

print()
show_role("SageMaker-AutoRole-Training")

  cleaned: SageMaker-AutoRole-Training


[06/10/26 12:21:43] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=867175;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=444787;file:///Users/nargokul/.pyenv/versions/sagemaker-recipes/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

[06/10/26 12:21:44] INFO     Found 1 MLflow apps: [('finetune-mlflow-1780611609', 'Created',  ]8;id=794590;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=890959;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py#134\134]8;;\
                             '3.10.1')]                                                                            

[06/10/26 12:21:45] INFO     Resolved MLflow app:                                             ]8;id=489113;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=176832;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/common_utils/finetune_utils.py#157\157]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:mlflow-app/app-X4PV75IB                      
                             XLBR (status: Created, version: 3.10.1)                                               

                    INFO     Resolved MLflow resource ARN:                                    ]8;id=960637;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py\base_evaluator.py]8;;\:]8;id=923565;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/base_evaluator.py#182\182]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:mlflow-app/app-X4PV75IB                      
                             XLBR                                                                                  

...

                    INFO     Updating pipeline resource.                                         ]8;id=185472;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=695713;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/resources.py#27255\27255]8;;\

[06/10/26 12:21:59] INFO     Successfully updated pipeline:                                        ]8;id=814984;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=907284;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/execution.py#254\254]8;;\
                             SagemakerEvaluation-BenchmarkEvaluation-668611b1-7c0e-4f9d-bc7d-34019                 
                             3190e91                                                                               

                    INFO     Starting pipeline execution: eval-meta-2e6f6dcc-1781119319            ]8;id=418300;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=700901;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/execution.py#308\308]8;;\

                    INFO     Pipeline execution started:                                           ]8;id=765174;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/execution.py\execution.py]8;;\:]8;id=23012;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-train/src/sagemaker/train/evaluate/execution.py#319\319]8;;\
                             arn:aws:sagemaker:us-west-2:211125564141:pipeline/SagemakerEvaluation                 
                             -BenchmarkEvaluation-668611b1-7c0e-4f9d-bc7d-340193190e91/execution/5                 
                             j988l8ktxol                                                                           

evaluate() started

✓ Role auto-created: SageMaker-AutoRole-Training
  Policies (6):
    - SageMaker-AutoRole-Training-cloudwatch_metric_policy
    - SageMaker-AutoRole-Training-cloudwatch_logs_policy
    - SageMaker-AutoRole-Training-ecr_policy
    - SageMaker-AutoRole-Training-ec2_policy
    - SageMaker-AutoRole-Training-kms_policy
    - SageMaker-AutoRole-Training-s3_policy


## 4. `ModelBuilder` — no role → auto-create serving role at construction
`ModelBuilder(...)` with no `role_arn` resolves the serving role in `_initialize_defaults`.

In [7]:
assert not role_exists("SageMaker-AutoRole-Serving")

from sagemaker.serve import ModelBuilder
from sagemaker.serve.mode.function_pointers import Mode

builder = ModelBuilder(
    model=MODEL_ID,
    mode=Mode.SAGEMAKER_ENDPOINT,
    # NOTE: no role_arn provided
)
print(f"builder.role_arn = {builder.role_arn}\n")
show_role("SageMaker-AutoRole-Serving")

[06/10/26 12:22:02] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=218427;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-serve/src/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=479599;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-serve/src/sagemaker/serve/model_builder_utils.py#340\340]8;;\

Model 'meta-textgeneration-llama-3-2-1b-instruct' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMetadata/eula/llama3_2Eula.txt for terms of use.


[06/10/26 12:22:04] INFO     Model 'meta-textgeneration-llama-3-2-1b-instruct' requires accepting      ]8;id=794507;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/jumpstart/utils.py\utils.py]8;;\:]8;id=636997;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/jumpstart/utils.py#647\647]8;;\
                             end-user license agreement (EULA). See                                                
                             https://jumpstart-cache-prod-us-west-2.s3.us-west-2.amazonaws.com/fmhMeta             
                             data/eula/llama3_2Eula.txt for terms of use.                                          

Using model 'meta-textgeneration-llama-3-2-1b-instruct' with wildcard version identifier '*'. You can pin to version '2.7.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


                    WARNING  Using model 'meta-textgeneration-llama-3-2-1b-instruct' with wildcard     ]8;id=965719;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/jumpstart/cache.py\cache.py]8;;\:]8;id=203957;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/jumpstart/cache.py#624\624]8;;\
                             version identifier '*'. You can pin to version '2.7.0' for more stable                
                             results. Note that models may have different input/output signatures                  
                             after a major version upgrade.                                                        

...

                    INFO     Created policy SageMaker-AutoRole-Serving-ecr_policy          ]8;id=209775;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=701914;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#144\144]8;;\

[06/10/26 12:22:06] INFO     Created policy                                                ]8;id=759397;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=338622;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#144\144]8;;\
                             SageMaker-AutoRole-Serving-cloudwatch_logs_policy                                     

                    INFO     Waiting 10s for IAM propagation...                            ]8;id=464410;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=605240;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#316\316]8;;\

[06/10/26 12:22:16] INFO     Using role:                                                   ]8;id=172471;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=427961;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#318\318]8;;\
                             arn:aws:iam::211125564141:role/SageMaker-AutoRole-Serving                             

builder.role_arn = arn:aws:iam::211125564141:role/SageMaker-AutoRole-Serving

✓ Role auto-created: SageMaker-AutoRole-Serving
  Policies (3):
    - SageMaker-AutoRole-Serving-s3_policy
    - SageMaker-AutoRole-Serving-ecr_policy
    - SageMaker-AutoRole-Serving-cloudwatch_logs_policy


## 5. `Pipeline` — no role → auto-create pipeline role on `.create()`
`Pipeline.create()` with no `role_arn` resolves the pipeline role. The create call then fails (no steps defined) **after** the role is created — expected.

In [8]:
assert not role_exists("SageMaker-AutoRole-Pipeline")

from sagemaker.mlops.workflow.pipeline import Pipeline

pipeline = Pipeline(name="iam-autocreate-test-pipeline")
try:
    pipeline.create()  # NOTE: no role_arn provided
    print("create() returned")
except Exception as e:
    print(f"(expected) create() failed downstream after role creation: {type(e).__name__}")

print()
show_role("SageMaker-AutoRole-Pipeline")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=397977;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=480431;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#110\110]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[06/10/26 12:22:17] INFO     Creating role 'SageMaker-AutoRole-Pipeline'...                ]8;id=587877;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=906383;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#303\303]8;;\

                    INFO     Created policy SageMaker-AutoRole-Pipeline-sagemaker_policy   ]8;id=499580;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=875602;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#144\144]8;;\

                    INFO     Created policy                                                ]8;id=477349;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=522781;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#144\144]8;;\
                             SageMaker-AutoRole-Pipeline-iam_passrole_policy                                       

                    INFO     Created policy SageMaker-AutoRole-Pipeline-s3_policy          ]8;id=31297;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=265633;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#144\144]8;;\

[06/10/26 12:22:18] INFO     Created policy                                                ]8;id=30799;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=401882;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#144\144]8;;\
                             SageMaker-AutoRole-Pipeline-cloudwatch_logs_policy                                    

                    INFO     Waiting 10s for IAM propagation...                            ]8;id=501792;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=524176;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#316\316]8;;\

[06/10/26 12:22:28] INFO     Using role:                                                   ]8;id=579573;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=38917;file:///Users/nargokul/workspace/nova-forge-integration/staging/notebooks/../sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#318\318]8;;\
                             arn:aws:iam::211125564141:role/SageMaker-AutoRole-Pipeline                            

(expected) create() failed downstream after role creation: ClientError

✓ Role auto-created: SageMaker-AutoRole-Pipeline
  Policies (4):
    - SageMaker-AutoRole-Pipeline-iam_passrole_policy
    - SageMaker-AutoRole-Pipeline-cloudwatch_logs_policy
    - SageMaker-AutoRole-Pipeline-sagemaker_policy
    - SageMaker-AutoRole-Pipeline-s3_policy


## Cleanup
Remove every auto-created role + policy and verify they are gone.

In [9]:
print("Removing auto-created roles...")
for r in ALL_ROLES:
    cleanup_role(r)

print("\nVerifying cleanup...")
all_gone = True
for r in ALL_ROLES:
    if role_exists(r):
        print(f"  ✗ {r} still exists!")
        all_gone = False
    else:
        print(f"  ✓ {r} deleted")

print("\n✓ ALL INTERFACES AUTO-CREATED ROLES — cleanup complete" if all_gone else "\n✗ Cleanup incomplete")

Removing auto-created roles...
  cleaned: SageMaker-AutoRole-Training
  cleaned: SageMaker-AutoRole-Serving
  cleaned: SageMaker-AutoRole-Pipeline

Verifying cleanup...
  ✓ SageMaker-AutoRole-Training deleted
  ✓ SageMaker-AutoRole-Serving deleted
  ✓ SageMaker-AutoRole-Pipeline deleted
  ✓ SageMaker-AutoRole-FeatureStore deleted

✓ ALL INTERFACES AUTO-CREATED ROLES — cleanup complete
